<a href="https://colab.research.google.com/github/joankl/Solar-Neutrinos-Project/blob/main/Solar-Neutrinos-Project%20/bis_data_candidates/lifetime_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Dedicated to Compute the Dataset Raw Lifetime using the runID lifetime calculator

The Lifetime should be calculated using the entery range of runIDs used for the analysis! No need to account for cuts, we are juts interested in the total lifetime of the available dataset. The cuts will be accountes in the selection efficiency based on MC.

In [2]:
import numpy as np
import pandas as pd
import glob
import pickle

# Load Files

## Load Hotspot results

In [7]:
# ==== Define the removal time in seconds ====
dt_hs_per_hs = 1

# ====== Load Dictionary ======

# Data directory pattern
fdir = '/home/joankl/data/solars/real_data/2p2PPO/HS_results/pkl_resume/hs_prompt_resume.pkl'

with open(fdir, 'rb') as f:
    hs_prompt_dict = pickle.load(f)
    N_hs = len(hs_prompt_dict['eventID'])

dt_hs_tot = N_hs * dt_hs_per_hs # in seconds

## Load the livetime list

In [46]:
#main_dir = 'C:/Users/joanc/jupyter notebooks/solar neutrino analysis/bis_data_candidates/run_lifetime_lists/*'
fdir = 'run_lifetime_list/livetime_per_run_LIVETIME_Bronze_300000_310637_april2025_v13.txt'

runID_list, time1_lt, time2_lt = [], [], []

with open(fdir, "r", encoding="utf-8") as f:
  for linea in f:
      if linea.strip():  # Evitar líneas vacías
          valores = linea.split()  # Separa por espacios o tabulaciones
          runID_list.append(int(valores[0]))
          time1_lt.append(float(valores[1]))
          time2_lt.append(float(valores[2]))

runID_list = np.array(runID_list)
time1_lt = np.array(time1_lt) # RAW LIVETIME (DAYS)
time2_lt = np.array(time2_lt) # NET LIVETIME (DAYS)

# RUN SELECTION:
runID_selection = (runID_list >= 301001)

runID_lt = runID_list[runID_selection]
time1_lt = np.array(time1_lt)[runID_selection] # RAW LIVETIME (DAYS)
time2_lt = np.array(time2_lt)[runID_selection] # NET LIVETIME (DAYS)

In [47]:
runID_lt

array([301001, 301002, 301003, ..., 310635, 310636, 310637], shape=(5712,))

## Load the Full Dataset runIDs

In [28]:
main_fdir = '/home/joankl/data/solars/real_data/2p2PPO/candidates/resume_files/'
fdir_list = glob.glob(main_fdir)

# Define the observables to grab from dataset: Save the into a dictionary
obs_list = ['runID', 'posr_av', 'energy_corrected']
obs_dict = {obs_i: np.array([]) for obs_i in obs_list}

for fdir_i in fdir_list:
    for var_i in obs_list:
        obs_i = np.load(fdir_i + var_i +'.npy')
        obs_dict[var_i] = np.append(obs_dict[var_i], obs_i)


runID_dataset = obs_dict['runID']
runID_dataset = np.unique(runID_dataset)

print(f'min. runID dataset: {np.min(runID_dataset)}')
print(f'max. runID dataset: {np.max(runID_dataset)}')

min. runID dataset: 300000.0
max. runID dataset: 310637.0


In [32]:
time2_lt.shape

(5712,)

# Compute the Dataset Livetime

In [53]:
# Extract the net lifetime (time2_lt) where runID from data is in runID_list
same_runID_condition = np.isin(runID_lt, runID_dataset)

filtered_lt = time2_lt[same_runID_condition]
dataset_lt_days_tot1 = np.sum(filtered_lt)

dataset_lf_days_tot2 = net_dataset_lt_days - dt_hs_tot / (24*3600)

print(f'Dataset with net livetime = {dataset_lt_days_tot1} days after RunID Selection')
print(f'Dataset with total livetime = {dataset_lf_days_tot2} days after RunID Selection and Hotspot cut')

Dataset with net livetime = 221.0155541425463 days after RunID Selection
Dataset with total livetime = 220.9362717351389 days after RunID Selection and Hotspot cut
